In [ ]:
# Libraries for data loading, tensor operations, and PyG heterogeneous graph modeling
import os
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv

In [ ]:
# Check that the extracted embedding and FIW relationship folders are available
print(os.listdir("data"))
print(os.listdir("data/fiw_embeddings")[:5])
print(os.listdir("data/FIDs")[:5])

['FIDs', 'fiw_embeddings']
['F0001', 'F0002', 'F0003', 'F0004', 'F0005']
['F0001', 'F0002', 'F0003', 'F0004', 'F0005']


In [ ]:
# Build a heterogeneous graph from all family graphs
# Node: each person (MID)
# Node feature: 512-dimensional ArcFace mean embedding
# Edge type: FIW relationship ID (RID), represented as rel_*
all_family_ids = os.listdir("data/fiw_embeddings")

x_list = []
global_node_to_idx = {}
node_family = []
edge_dict = defaultdict(list)

node_offset = 0

for family_id in all_family_ids:
    family_path = f"data/fiw_embeddings/{family_id}"
    mid_csv_path = f"data/FIDs/{family_id}/mid.csv"

    if not os.path.exists(mid_csv_path):
        continue

    local_node_to_idx = {}

    # 1) nodes
    # Add person nodes and load their mean face embeddings
    for mid in os.listdir(family_path):
        emb_path = os.path.join(family_path, mid, "mean_embedding.pkl")

        if os.path.exists(emb_path):
            with open(emb_path, "rb") as f:
                emb = pickle.load(f)

            x_list.append(emb)
            local_node_to_idx[mid] = node_offset
            global_node_to_idx[(family_id, mid)] = node_offset
            node_family.append(family_id)
            node_offset += 1

    # 2) Add typed edges using the FIW relationship matrix
    # Each nonzero RID becomes a separate HGT edge type
    df = pd.read_csv(mid_csv_path)

    for _, row in df.iterrows():
        src_mid = f"MID{int(row['MID'])}"

        for col in df.columns:
            if col.isdigit():
                rid = int(row[col])

                if rid != 0:
                    dst_mid = f"MID{int(col)}"

                    if src_mid in local_node_to_idx and dst_mid in local_node_to_idx:
                        u = local_node_to_idx[src_mid]
                        v = local_node_to_idx[dst_mid]

                        edge_type = f"rel_{rid}"
                        edge_dict[edge_type].append([u, v])

In [ ]:
# Convert collected nodes and typed edges into PyG HeteroData format
# HGT requires edge types such as ("person", "rel_4", "person")
data = HeteroData()

x = torch.tensor(np.array(x_list), dtype=torch.float).squeeze(1)
data["person"].x = x

for rel, edges in edge_dict.items():
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    data["person", rel, "person"].edge_index = edge_index

node_family = np.array(node_family)

print(data)
print(data.metadata())

HeteroData(
  person={ x=[5773, 512] },
  (person, rel_5, person)={ edge_index=[2, 2232] },
  (person, rel_4, person)={ edge_index=[2, 4758] },
  (person, rel_1, person)={ edge_index=[2, 4760] },
  (person, rel_2, person)={ edge_index=[2, 4770] },
  (person, rel_3, person)={ edge_index=[2, 1147] },
  (person, rel_6, person)={ edge_index=[2, 1148] },
  (person, rel_8, person)={ edge_index=[2, 105] },
  (person, rel_7, person)={ edge_index=[2, 103] }
)
(['person'], [('person', 'rel_5', 'person'), ('person', 'rel_4', 'person'), ('person', 'rel_1', 'person'), ('person', 'rel_2', 'person'), ('person', 'rel_3', 'person'), ('person', 'rel_6', 'person'), ('person', 'rel_8', 'person'), ('person', 'rel_7', 'person')])


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HGTConv

# HGT encoder:
# Learns relation-aware person embeddings using typed kinship edges
# Output dimension is set to 128 for compact indexing vectors
class HGTEncoder(nn.Module):
    def __init__(self, metadata, hidden_channels=128, heads=2):
        super().__init__()

        self.conv1 = HGTConv(
            in_channels=-1,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads
        )

        self.conv2 = HGTConv(
            in_channels=hidden_channels,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads
        )

    def forward(self, x_dict, edge_index_dict):
        # First HGT layer aggregates typed neighbor information
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: F.relu(x) for key, x in x_dict.items()}

        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict

In [ ]:
import random

# Node-level contrastive loss for indexing:
# Pull embeddings from the same family closer
# Push embeddings from different families farther apart
def contrastive_loss(z, node_family, num_samples=300):
    loss = torch.tensor(0.0, device=z.device)

    for _ in range(num_samples):
        i = random.randint(0, z.size(0) - 1)

        same_family = np.where(node_family == node_family[i])[0]
        diff_family = np.where(node_family != node_family[i])[0]

        j = int(np.random.choice(same_family))
        k = int(np.random.choice(diff_family))

        pos_sim = F.cosine_similarity(z[i], z[j], dim=0)
        neg_sim = F.cosine_similarity(z[i], z[k], dim=0)

        loss += -torch.log(torch.sigmoid(pos_sim - neg_sim) + 1e-8)

    return loss / num_samples

In [ ]:
# Train HGT encoder with the contrastive objective
# The output z["person"] contains relation-aware embeddings for all person nodes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = HGTEncoder(
    metadata=data.metadata(),
    hidden_channels=128,
    heads=2
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()

for epoch in range(100):
    optimizer.zero_grad()

    out_dict = model(data.x_dict, data.edge_index_dict)
    z = out_dict["person"]

    loss = contrastive_loss(z, node_family, num_samples=800)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

cuda
Epoch 0, Loss: 0.6925
Epoch 10, Loss: 0.6564
Epoch 20, Loss: 0.5810
Epoch 30, Loss: 0.5511
Epoch 40, Loss: 0.5291
Epoch 50, Loss: 0.5070
Epoch 60, Loss: 0.4767
Epoch 70, Loss: 0.4799
Epoch 80, Loss: 0.4585
Epoch 90, Loss: 0.4577


In [17]:
#print(data["person"].x.device)
#print(next(model.parameters()).device)
#print(data)

In [ ]:
# Save final HGT embeddings and metadata for downstream vector search
model.eval()
with torch.no_grad():
    out_dict = model(data.x_dict, data.edge_index_dict)
    z = out_dict["person"]

torch.save({
    "embeddings": z.cpu(),
    "node_to_idx": global_node_to_idx,
    "node_family": node_family,
    "metadata": data.metadata(),
}, "final_hgt_index_embeddings.pt")

print(z.shape)
print("saved final_hgt_index_embeddings.pt")

torch.Size([5773, 128])
saved final_hgt_index_embeddings.pt


In [ ]:
# Simple retrieval sanity check:
# Query one node and retrieve the nearest nodes by cosine similarity
import torch.nn.functional as F

z = z.cpu()

i = 0  # query node index for sanity check
sim = F.cosine_similarity(z[i].unsqueeze(0), z)

topk = torch.topk(sim, k=10)

print("Query family:", node_family[i])
print("Top results families:")
for idx in topk.indices:
    print(node_family[idx.item()])

Query family: F0001
Top results families:
F0001
F0001
F0137
F0105
F0137
F0187
F0187
F0059
F0112
F0059
